# 🎙️ أصوات ذكاء اصطناعي لـ«مدرسة البيت»
يولّد هذا الدفتر مقاطع صوتية طبيعية بالعربي والإنجليزي لكل كلمة وسؤال في الموقع، ثم يبني ملف `index.html` جديد فيه الأصوات.

**قبل البدء:** من القائمة العلوية: **Runtime ← Change runtime type ← T4 GPU** ثم Save.

بعدها شغّل الخلايا بالترتيب (زر ▶ على يسار كل خلية).

## ١) تثبيت الأدوات (٢–٤ دقائق)

In [ ]:
!apt-get -qq install -y ffmpeg nodejs > /dev/null
!pip -q install silma-tts
print('✅ تم التثبيت')

## ٢) ارفع ملف المشروع `madrasat-albayt.zip`
اضغط **Choose Files** واختر الملف.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()
name = list(up)[0]
zipfile.ZipFile(name).extractall('/content')
os.chdir('/content/madrasat-albayt')
print('✅ المشروع جاهز:', os.getcwd())

## ٣) الصوت الذي سيُقلَّد
الذكاء الاصطناعي **يقلّد صوتًا** من تسجيل قصير. اختر **واحدًا**:

- **أ) صوتك أنت** (ممتع للأطفال!): سجّل ٥–٨ ثوانٍ بالجوال في مكان هادئ، تتكلم بالفصحى بوضوح، مثل: «مرحبًا يا أبطال، هيا نتعلّم معًا الحروف والأرقام والألوان». ثم شغّل الخلية **أ**. أي صيغة تعمل (m4a, mp3, wav…).
- **ب) صوت جاهز** من مشروع SILMA: شغّل الخلية **ب**.

صوت واحد عربي يكفي، فالنموذج يتكلم الإنجليزي بالصوت نفسه.

In [ ]:
# أ) ارفع تسجيلك (يمكن رفع ملف المذكرات الصوتية من الآيفون .m4a مباشرة)
import IPython.display as d
up = files.upload()
src = list(up)[0]
!ffmpeg -loglevel error -y -i "{src}" -ac 1 -ar 24000 -t 9 tools/voices/ar.wav
!rm -f tools/voices/en.wav
print('✅ تم حفظ صوتك. استمع للتأكد أنه صوتك:')
d.display(d.Audio('tools/voices/ar.wav'))

In [ ]:
# ب) صوت جاهز من SILMA (صوت نسائي) – لا تشغّل هذه الخلية إذا استخدمت صوتك في (أ)، لأنها تستبدله
!git clone -q --depth 1 https://github.com/SILMA-AI/silma-tts /content/silma-src
import glob, shutil
s = sorted(glob.glob('/content/silma-src/**/ref_audio_samples/*.wav', recursive=True))
print(s)
ar = [x for x in s if '/ar' in x.lower()] or s
shutil.copy(ar[0], 'tools/voices/ar.wav')
en = [x for x in s if '/en' in x.lower()]
if en: shutil.copy(en[0], 'tools/voices/en.wav')
print('✅ تم اختيار الصوت')

## ٤) جرّب ١٠ جمل أولًا واسمعها

In [ ]:
!PYTHONWARNINGS=ignore python tools/make_audio.py --limit 10
import json, IPython.display as d
m = json.load(open('src/audio/manifest.json', encoding='utf-8'))
for k, f in list(m.items())[:5]:
    print(k); d.display(d.Audio('src/audio/clips/' + f))

إذا أعجبك الصوت أكمل. إذا لا: أعد الخطوة ٣ بتسجيل آخر ثم أعد الخطوة ٤؛ المقاطع القديمة تُحذف تلقائيًا عند تغيير الصوت.

## ٥) توليد كل الأصوات (حوالي ٢٠–٤٠ دقيقة على T4)

In [ ]:
!PYTHONWARNINGS=ignore python tools/make_audio.py

## ٦) بناء الموقع وتنزيله
سينزل ملفان: `index.html` (الموقع مع الأصوات) و`madrasat-albayt.zip` (المشروع محدّثًا بالأصوات، احتفظ به للمرات القادمة).

In [ ]:
!python build.py
!cd /content && rm -f madrasat-albayt.zip && zip -qr madrasat-albayt.zip madrasat-albayt
files.download('dist/index.html')
files.download('/content/madrasat-albayt.zip')

**عند إضافة وحدة جديدة لاحقًا:** ارفع الـ zip المحدّث وأعد الخطوات ١ و٢ و٥ و٦ فقط. سيولّد الجمل الجديدة فقط.